# MediScanX: End-to-End CXR Diagnostic Pipeline
**Project:** AI-Powered Multimodal Medical Scanner.<br>
**Objective:** Train the CIHMLC DenseNet121 model on the CheXpert dataset.<br> 
**Infrastructure:** This notebook serves as the cloud-GPU execution environment. Stable classes developed here will be migrated to the local `ml_pipeline/src/` repository.<br>

---

## Section 1: Global Environment & Initialization
Setting up the GPU accelerator, defining constants, and importing core libraries for tensor manipulation and computer vision.

In [3]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("Libraries Imported Successfully.")

Libraries Imported Successfully.


## Section 2: Centralized Configuration
**Target Repository File:** `ml_pipeline/src/configs/cxr_config.py`

Defines all hyperparameters, architectural flags, and hardware settings. Modifying this single dataclass dictates the behavior of the entire pipeline.

In [19]:
from dataclasses import dataclass, asdict
import torch

@dataclass
class CXRConfig:
    """Centralized configuration for the Chest X-Ray CIHMLC pipeline."""
    # Project Info
    project_name: str = 'MediScanX'
    run_name: str = 'DenseNet121-CIHMLC-70-15-15-test-run-cpu'

    # Hardware and Data Paths
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    kaggle_data_root: str = "/kaggle/input/datasets/ashery/chexpert"
    csv_train_path: str = f"{kaggle_data_root}/train.csv"
    
    # DataLoader Configs
    batch_size: int = 32
    num_workers: int = 4
    image_size: tuple = (320, 320)
    
    # Model Architecture
    backbone: str = 'DenseNet121'
    num_classes: int = 14
    pretrained: bool = True
    
    # Training Hyperparameters
    epochs: int = 40
    learning_rate: float = 1e-4
    patience: int = 5
    penalty_weight: float = 1.5
    
    # Data Split Parameters
    train_size: float = 0.7
    val_size: float = 0.15
    test_size: float = 0.15
    random_seed: int = 42

# Instantiate the global configuration
CFG = CXRConfig()
print(f"Executing MediScanX CXR Pipeline on: {CFG.device}")

Executing MediScanX CXR Pipeline on: cpu


## Section 3: Radiographic Preprocessing & Data Ingestion
**Target Repository File:** `ml_pipeline/src/data/transforms.py`

This section implements the localized preprocessing pipeline. We utilize Contrast Limited Adaptive Histogram Equalization (CLAHE) to standardize illumination and enhance lung field contrast, removing biases caused by legacy X-ray machines.

In [20]:
import cv2
import numpy as np
from torchvision import transforms

class ApplyCLAHE(object):
    """
    A custom PyTorch transform that applies Contrast Limited Adaptive Histogram Equalization (CLAHE) to a medical image.
    
    This class standardizes radiographic inputs by normalizing illumination discrepancies and enhancing local contrast 
    within lung fields, mitigating artifacts  from legacy X-ray machines.
    
    Attributes:
        clip_limit(float): Threshold for contrast limiting.
        tile_grid_size (tuple): Size of the grid for histogram equalization.
        clahe(cv2.CLAHE): The instantiated OpenCV CLAHE object.
    """
    
    def __init__(self, clip_limit: float=2.0, tile_grid_size: tuple=(8,8)):
        """
        Initializes the ApplyCLAHE transform object.

        Args:
            clip_limit (float, optional): Sets the threshold for contrast limiting. Defaults to 2.0.
            tile_grid_size (tuple, optional): Sets the grid size for localized equalization. Defaults to (8,8).
        """
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
        self.clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        
    def __call__(self, img: np.ndarray) -> np.ndarray:
        """
        Executes CLAHE and stacks the output into a 3-channel array for transfer learning.

        Args:
            img (np.ndarray): The input image array.

        Returns:
            np.ndarray: The contrast-enhanced, 3-channel image array.
            
        Raises:
            TypeError: If the input is not a NumPy array. 
        """
        if not isinstance(img, np.ndarray):
            raise TypeError(f"ApplyCLAHE expects a numpy.ndarray, but got{type(img)}")
        
        # Ensure the image is 8-bit grayscale as required by OpenCV CLAHE
        if img.dtype != np.uint8:
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
            img = img.astype(np.uint8)
            
            
        # Apply the CLAHE algorithm
        enhanced_img = self.clahe.apply(img)
        enhanced_img_3c = cv2.cvtColor(enhanced_img, cv2.COLOR_GRAY2RGB)
        
        return enhanced_img_3c
    
    def __repr__(self) -> str:
        """Return a string representation of the transform object."""
        return f"{self.__class__.__name__}(clip_limit={self.clip_limit}, tile_grid_size={self.tile_grid_size})"
    
class RadiographicPipeline:
    """
    A factory class to construct the complete preprocessing pipeline for Chest X-Rays prior to model ingestion.
    """
    @staticmethod
    def get_training_transforms(resize_dim: tuple=(320, 320)) -> transforms.Compose:
        """
        Constructs the sequence of transforms including CLAHE, resizing and tensor conversion.
        Args:
            resize_dim (tuple, optional): Target dimensions for the neural network. Defaults to (320,320).

        Returns:
            transforms.Compose: The chained PyTorch transformations.
        """
        return transforms.Compose([
            ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8)),
            transforms.ToPILImage(),
            transforms.Resize(resize_dim),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
            # Normalize using full 3-channel ImageNet stats
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

**Target Repository File:** `ml_pipeline/src/data/chexpert_dataset.py`

Defining the `CheXpertDataset` class. 
*Clinical Note:* To prioritize sensitivity in a triage environment, we are adopting the **U-Ones policy**. All ambiguous/uncertain diagnoses (-1) in the dataset are mathematically mapped to positive (1) to prevent potentially fatal false negatives.

In [6]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class CheXpertDataset(Dataset):
    """
    A custom PyTorch Dataset implementation for CheXpert database.
    
    This class handles the parsing of the clinical annotations in CSV, loads the corresponding high-resolution
    X-ray images from the disk using OpenCV, applies the defined preprocessing transforms, and
    extracts the multi-label pathology vectors.

    Attributes:
        annotations (pd.DataFrame): The parsed CSV data containing paths and clinical labels.
        root_dir(str): The base directory path where the image dataset is stored.
        transform(callable, optional): A pytorch transform to apply to the images.
    """
    def __init__(self, csv_file: str, root_dir: str, transform: callable=None):
        """
        Initializes the CheXpert Dataset.

        Args:
            csv_file (str): Path to the train.csv or valid.csv file.
            root_dir (str): Directory containing the CheXpert image folders.
            transform (callable, optional): Optional transform to be applied on a sample. Defaults to None.
        """
        super().__init__()
        # Load the CSV. CheXpert labels sometimes contain -1 (uncertain).
        # U-Ones Policy: Map -1(uncertain) to 1 (positive) for triage safety.
        self.annotations = pd.read_csv(csv_file)
        self.annotations = self.annotations.fillna(0)
        self.annotations = self.annotations.replace(-1, 1)
        
        self.root_dir = root_dir
        self.transform = transform
        
    def __len__(self) -> int:
        """Returns the total number of patient scans in the dataset."""
        return len(self.annotations)
    
    def __getitem__(self, idx: int) -> tuple:
        """
        Retrieves a single medical image and its corresponding pathology labels.

        Args:
            idx (int): The index of the item to retrieve.

        Returns:
            tuple: A tuple containing (image_tensor, label_tensor).
        """
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        # The CheXpert 'Path column contains the relative path
        original_path = self.annotations.iloc[idx]['Path']
        cleaned_path = original_path.replace('CheXpert-v1.0-small/', '')
        img_name = os.path.join(self.root_dir, cleaned_path)
        
        # Load the image strictly in grayscale
        image = cv2.imread(img_name, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            raise FileNotFoundError(f"OpenCV could not read the image at {img_name}")

        # Extract the 14 pathology labels (columns 5 to 18 in CheXpert)
        labels = self.annotations.iloc[idx, 5:19].values.astype(np.float32)
        
        if self.transform:
            image = self.transform(image)
            
        labels = torch.tensor(labels)
        
        return image, labels

print("CheXpert Dataset class defined successfully.")

CheXpert Dataset class defined successfully.


## Section 3.1: Pipeline Verification
Testing the DataLoader to ensure the dataset builds correctly, tensors are shaped as expected, and the U-Ones policy is active without crashing the memory.

In [7]:
if os.path.exists(CFG.csv_train_path):
    # Initialize the transforms using CFG
    cxr_transforms = RadiographicPipeline.get_training_transforms(resize_dim=CFG.image_size)
    
    # Instantiate the Dataset using CFG
    train_dataset = CheXpertDataset(
        csv_file=CFG.csv_train_path, 
        root_dir=CFG.kaggle_data_root, 
        transform=cxr_transforms
    )
    
    # Instantiate the DataLoader using CFG
    train_loader = DataLoader(dataset=train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
    
    data_iterator = iter(train_loader)
    images, labels = next(data_iterator)
    
    print(f"Data Pipeline Verified!")
    print(f"Image Tensor Shape: {images.shape} | Expected: [{CFG.batch_size}, 1, {CFG.image_size[0]}, {CFG.image_size[1]}]") 
    print(f"Labels Tensor Shape: {labels.shape} | Expected: [{CFG.batch_size}, {CFG.num_classes}]")
else:
    print(f"Dataset missing at {CFG.csv_train_path}. Please attach CheXpert to the kernel.")

Data Pipeline Verified!
Image Tensor Shape: torch.Size([32, 1, 320, 320]) | Expected: [32, 1, 320, 320]
Labels Tensor Shape: torch.Size([32, 14]) | Expected: [32, 14]


## Section 4: CIHMLC Model Architecture (DenseNet121)
**Target Repository File:** `ml_pipeline/src/models/densenet121_cihmlc.py`

This section defines the core computer vision architecture. We instantiate a pretrained DenseNet121 and strip its default classifier. We then append a custom `Conv2d` layer (512 filters) for finer structural extraction, followed by Global Average Pooling (GAP). 

**Architectural Imperative:** The `forward` method is explicitly restructured to return a tuple: `(logits, spatial_feature_maps)`. This multi-output design is a strict requirement for the on-device Grad-CAM++ explainability workaround later in the pipeline.

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

class DenseNet121_CIHMLC(nn.Module):
    """
    Clinically-Inspired Hierarchical Multi-Label Classification Model.
    
    Utilizes a DenseNet121 backbone optimized for feature reuse. The architecture is
    heavily modified with a custom convolutional head and a dual-output forward pass
    to enable offline Explainable AI (Grad-CAM++) on mobile edge devices.

    Args:
        features (nn.Sequential): The pretrained DenseNet121 feature extractor.
        custom_conv (nn.Conv2d): Dimensionality reduction and fine structural extraction layer.
        relu (nn.ReLU): Non-linear activation for the custom conv layer.
        global_avg_pool (nn.AdaptiveAvgPool2d): GAP layer to preserve spatial context.
        classifier (nn.Linear): The final dense layer mapping to the 14 clinical pathologies.
    """
    def __init__(self, num_classes: int=14, pretrained: bool=True):
        """
        Initializes the DenseNet121_CIHMLC architecture.

        Args:
            num_classes (int, optional): The number of output diagnostic labels. Defaults to 14.
            pretrained (bool, optional): Whether to initialize the ImageNet weights. Defaults to True.
        """
        super(DenseNet121_CIHMLC, self).__init__()
        
        # Load the foundation DenseNet121 backbone
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        densenet = models.densenet121(weights=weights)
        
        # Extract the feature blocks
        self.features = densenet.features
        
        # Append Custom Conv2D Layer
        # DenseNet121's feature extractor naturally outputs 1024 channels.
        # We compress this to 512 filters to extract finer structural details and reduce parameters
        self.custom_conv = nn.Conv2d(
            in_channels=1024,
            out_channels=512,
            kernel_size=3,
            padding=1,
            bias=False
        )
        # BatchNorm and ReLU for stabilization and non-linearity
        self.bn = nn.BatchNorm2d(512)
        self.relu = nn.ReLU(inplace=True)
        
        # Global Average Pooling: Condenses the spatial dimensions (H,W) to (1,1) while preserving the 512 feature maps
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Final Classification layer
        self.classifier = nn.Linear(in_features=512, out_features=num_classes)
        
    def forward(self, x: torch.Tensor) -> tuple:
        """
        Executes the dual-output forward pass.

        Args:
            x (torch.Tensor): A batch of 3-channel X-ray images. Shape: [B, 3, H, W]

        Returns:
            tuple: 
                - logits (torch.Tensor): The raw, unactivated classification scores. Shape: [B, 14]
                - spatial_features (torch.Tensor): The raw spatial maps for Grad-CAM++. Shape: [B, 512, H', W']
        """
        # Pass through the deep DenseNet blocks
        x = self.features(x)
        
        # Pass through the custom structural extraction head
        spatial_features = self.relu(self.bn(self.custom_conv(x)))
        
        # Pool, flatten and classify
        pooled = self.global_avg_pool(spatial_features)
        flattened = torch.flatten(pooled, 1)
        logits = self.classifier(flattened)
        
        # Return both artifacts for the inference engine
        return logits, spatial_features

print("DenseNet121 CIHMLC Model Architecture defined.")

DenseNet121 CIHMLC Model Architecture defined.


## Section 4.1: Architectural Verification
Testing the `DenseNet121_CIHMLC` instantiation. We will pass a dummy tensor formatted exactly like our `DataLoader` output `[Batch, Channels, Height, Width]` to ensure the dual-output tuple mathematically aligns with our expectations.

In [9]:
# Instantiate the model using CFG
model = DenseNet121_CIHMLC(num_classes=CFG.num_classes, pretrained=CFG.pretrained).to(CFG.device)

# Create a dummy batch using CFG dimensions
dummy_batch = torch.randn(CFG.batch_size, 3, CFG.image_size[0], CFG.image_size[1]).to(CFG.device)

logits, spatial_maps = model(dummy_batch)

print("Architecture Verification Successful!")
print(f"Logits Shape: {logits.shape} | Expected: [{CFG.batch_size}, {CFG.num_classes}]")
print(f"Spatial Feature Maps Shape: {spatial_maps.shape} | Expected: [{CFG.batch_size}, 512, 10, 10]")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

Architecture Verification Successful!
Logits Shape: torch.Size([32, 14]) | Expected: [32, 14]
Spatial Feature Maps Shape: torch.Size([32, 512, 10, 10]) | Expected: [32, 512, 10, 10]
Total Trainable Parameters: 11,680,654


## Section 5: Clinical Taxonomy & Imbalance Weighting
**Target Repository File:** `ml_pipeline/src/utils/metrics.py`

This section defines the hierarchical relationships between pathologies based on the CheXpert labeling schema. It also calculates class-specific positive weights to mathematically counter the severe class imbalances in the dataset.

In [22]:
import torch
import numpy as np
import pandas as pd
# Define the standard CheXpert labels in order
CHEXPERT_LABELS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis", 
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"
]

# Map child indices to their parent disease index.
# CheXpert Label Indices (Matching the Dataset class order):
# 0: No Finding, 1: Enlarged Cardiomediastinum, 2: Cardiomegaly
# 3: Lung Opacity, 4: Lung Lesion, 5: Edema, 6: Consolidation
# 7: Pneumonia, 8: Atelectasis, 9: Pneumothorax, 10: Pleural Effusion
# 11: Pleural Other, 12: Fracture, 13: Support Devices

# Define the Clinical Taxonomy (Parent Index -> List of Child Indices)
# Based on the CheXpert label indices:
# 1: 'Enlarged Cardiomediastinum' -> 2: 'Cardiomegaly'
# 3: 'Lung Opacity' -> 4: 'Lung Lesion', 5: 'Edema', 6: 'Consolidation', 8: 'Atelectasis
# 6: 'Consolidation -> 7: 'Pneumonia'

# List of Tuples (parent_index, child_index)
# Enforces multi-level hierarchical rules
HIERARCHY_PAIRS = [
    (1, 2),
    (3, 4),
    (3, 5),
    (3, 6),
    (3, 8),
    (6, 7)
]

class ClassWeightCalculator:
    """Utility class to calculate positive weights for imbalanced datasets."""
    
    @staticmethod
    def compute_pos_weights(df: pd.DataFrame, num_classes: int = 14) -> torch.Tensor:
        """
        Calculates the ratio of negative to positive samples for each class.
        
        Args:
            df (pd.DataFrame): The training dataframe.
            num_classes (int): Total number of pathology labels.
            
        Returns:
            torch.Tensor: A 1D tensor of positive weights for BCEWithLogitsLoss.
        """
        # Clean the dataframe dynamically: fill NaNs with 0, and apply U-Ones policy (-1 to 1)
        df_clean = df.fillna(0).replace(-1, 1)
        
        # Extract the label matrix (assuming CheXpert labels start at column index 5)
        labels = df_clean.iloc[:, 5:5+num_classes].values
        
        pos_counts = np.sum(labels == 1, axis=0)
        neg_counts = np.sum(labels == 0, axis=0)
        
        # Add a small epsilon to prevent division by zero
        pos_weights = neg_counts / (pos_counts + 1e-7)
        return torch.tensor(pos_weights, dtype=torch.float32)

print("Defined Clinical Taxonomies and Weight Calculator.")

Defined Clinical Taxonomies and Weight Calculator.


## Section 6: The Hierarchical Loss Function (HBCE)
**Target Repository File:** `ml_pipeline/src/models/losses.py`

The `HBCELoss` combines standard weighted BCE with a custom penalty mechanism. It iterates through the defined `HIERARCHY_PAIRS`. If the model assigns a higher probability to a child class than its mandatory parent class, a mathematical penalty is applied, forcing the neural network to learn anatomically correct associations.

In [11]:
import torch.nn as nn
import torch.nn.functional as F

class HBCELoss(nn.Module):
    """
    Hierarchical Binary Cross-Entropy Loss.
    
    Attributes:
        pos_weight (torch.Tensor): Weights to counter class imbalance.
        hierarchy_pairs (list): List of (parent_idx, child_idx) tuples.
        penalty_weight (float): Multiplier for the hierarchical violation penalty.
    """
    
    def __init__(self, pos_weight: torch.Tensor, hierarchy_pairs: list, penalty_weight: float = 1.0):
        super(HBCELoss, self).__init__()
        self.pos_weight = pos_weight
        self.hierarchy_pairs = hierarchy_pairs
        self.penalty_weight = penalty_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Calculates the penalized loss using the cascading taxonomy map.
        """
        # 1. Standard Weighted Binary Cross Entropy
        bce_loss = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='mean'
        )
        
        # 2. Compute probabilities for the hierarchy check
        probs = torch.sigmoid(logits)
        
        # 3. Calculate Hierarchical Penalties using the provided tuples
        penalty = torch.tensor(0.0, device=logits.device)
        
        for parent_idx, child_idx in self.hierarchy_pairs:
            # Violation occurs if P(Child) > P(Parent)
            # ReLU zeros out the tensor if P(Parent) is correctly higher than P(Child)
            violation = F.relu(probs[:, child_idx] - probs[:, parent_idx])
            penalty += torch.mean(violation)
            
        # 4. Total Loss computation
        total_loss = bce_loss + (self.penalty_weight * penalty)
        return total_loss

print("Defined the custom Hierarchical Loss function based on Hierarchical Binary Cross-Entropy.")

Defined the custom Hierarchical Loss function based on Hierarchical Binary Cross-Entropy.


## Section 7: The Model Trainer
**Target Repository File:** `ml_pipeline/src/utils/trainer.py`

This execution engine manages the DenseNet121 orchestration. It handles forward passes, complex multi-label metric calculation (Loss, AUC, F1, Precision, Recall) across Train, Validation, and Test sets, and enforces Early Stopping with real-time Weights & Biases (WandB) synchronization.

In [23]:
import torch
import numpy as np
import wandb
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

class CIHMLCTrainer:
    """
    Trainer managing model orchestration, metric evaluation, and experiment
    tracking via Weights and Biases.
    
    Attributes:
        model (torch.nn.Module): The PyTorch neural network to be trained.
        train_loader (DataLoader): Iterable over the training dataset.
        val_loader (DataLoader): Iterable over the validation dataset.
        criterion (torch.nn.Module): The loss function (e.g. HBCELoss).
        optimizer (torch.optim.Optimizer): The optimizatoin algorithm.
        scheduler (torch.optim.lr_scheduler): Learning rate decay scheduler.
        cfg (CXRConfig): The centralized configuration object.
        best_auc (float): Tracks the highest achieved validation AUC. 
    """
    def __init__(self, model, train_loader, val_loader, test_loader, criterion, optimizer, scheduler, config):
        """Initializes the CIHMLCTrainer with the require PyTorch components and configs."""
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.cfg = config
        self.best_auc = 0.0

    def _calculate_metrics(self, targets: np.ndarray, probs: np.ndarray, preds: np.ndarray) -> tuple:
        """Internal helper to calculate macro-averaged classification metrics."""
        try: 
            mean_auc = float(roc_auc_score(targets, probs, average='macro'))
            f1 = float(f1_score(targets, probs, average='macro', zero_division=0))
            precison = float(precision_score(targets, probs, average='macro', zero_division=0))
            recall = float(recall_score(targets, probs, average='macro', zero_division=0))
        except ValueError:
            # Fallback if a batch lacks positive samples for minority classes
            mean_auc, f1, precision, recall = 0.0, 0.0, 0.0, 0.0
        
        return mean_auc, f1, precision, recall
    
    def train_epoch(self) -> float:
        """
        Executes one complete pass over the training dataset.

        Returns:
            tuple: Contains (train_loss, train_auc, train_f1, train_precision, train_recall)
        """
        self.model.train()
        running_loss = 0.0
        all_targets, all_probs, all_preds = [], [], []
        
        loop = tqdm(self.train_loader, desc='Training', leave=False)
        for images, labels in loop:
            images, labels = images.to(self.cfg.device), labels.to(self.cfg.device)
            
            self.optimizer.zero_grad()
            logits, _ = self.model(images)
            loss = self.criterion(logits, labels)
            
            loss.backward()
            self.optimizer.step()
            running_loss += loss.item()
            loop.set_postfix(loss=loss.item())
        
        # Store predictions for epoch-level metric calculation
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            all_targets.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            
        epoch_loss = running_loss / len(self.train_loader)
        auc, f1, precision, recall = self._calculate_metrics(all_targets, all_probs, all_preds)

        return epoch_loss, auc, f1, precision, recall
        
    def evaluate(self, dataloader: DataLoader, desc: str ='validating') -> tuple:
        """
        Evaluates the model on the provided dataloader.

        Args:
            dataloader (DataLoader): The validation or the test dataset loader.
            desc (str, optional): The description for the tqdm progress bar. Defaults to 'validating'.

        Returns:
            tuple: Contains (loss, macro_auc, macro_f1, macro_precision, macro_recall).
        """
        self.model.eval()
        total_loss = 0.0
        all_targets, all_probs, all_preds = [], [], []
        
        with torch.no_grad():
            for images, labels in tqdm(dataloader, desc=desc, leave=False):
                images, labels = images.to(self.cfg.device), labels.to(self.cfg.device)
                
                logits, _ = self.model(images)
                loss = self.criterion(logits, labels)
                total_loss += loss.item()
                
                # Convert raw logits to probabilites and binary predictions (0.5 threshold)
                probs = torch.sigmoid(logits)
                preds = (probs > 0.5).float()
                
                all_targets.append(labels.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                
        all_targets = np.vstack(all_targets)
        all_probs = np.vstack(all_probs)
        all_preds = np.vstack(all_preds)
            
        epoch_loss = total_loss / len(dataloader)
        auc, f1, precision, recall = self._calculate_metrics(all_targets, all_probs, all_preds)
        
        return epoch_loss, auc, f1, precision, recall
    
    def execute_training(self, save_path: str='best_densenet_cihmlc.pth') -> None:
        """
        Orchestrates the training loop, early stopping and WandB telemetry logging.
        Automatically executes final test-set evaluation upon completion.

        Args:
            save_path (str, optional): Filepath to save the best model weights. Defaults to 'best_densenet_cihmlc.pth'.
        """
        patience_counter = 0
        
        for epoch in range(self.cfg.epochs):
            print(f"\n========== Epoch {epoch+1}/{self.cfg.epochs} ===========")
            
            # Train and evaluate
            t_loss, t_auc, t_f1, t_precision, t_recall = self.train_epoch()
            v_loss, v_auc, v_f1, v_precision, v_recall = self.evaluate(self.val_loader, desc='Validating')

            # Step Scheduler
            self.scheduler.step(v_loss)
            current_lr = self.optimizer.param_groups[0]['lr']

            # WandB Logging (per Epoch)
            wandb.log({
                "epoch": epoch + 1,
                "learning_rate": current_lr,
                "train/loss": t_loss, "train/auc": t_auc, "train/f1": t_f1, "train/precision": t_precision, "train/recall": t_recall,
                "val/loss": v_loss, "val/auc": v_auc, "val/f1": v_f1, "val/precision": v_precision, "val/recall": v_recall,
            })
            
            print(f"TRAIN -> Loss: {t_loss:.4f} | AUC: {t_auc:.4f} | F1: {t_f1} | Precision: {t_precision:.4f} | Recall: {t_recall:.4f}")
            print(f"VALID -> Loss: {v_loss:.4f} | AUC: {v_auc:.4f} | F1: {v_f1} | Precision: {v_precision:.4f} | Recall: {v_recall:.4f}")
            
            # Checkpointing and Early Stopping
            if v_auc > self.best_auc:
                self.best_auc = v_auc
                torch.save(self.model.state_dict(), save_path)
                print(f"*** New Best Model Saved (Val AUC: {v_auc:.4f}) ***")
                patience_counter = 0
            else:
                patience_counter += 1
                
            if patience_counter >= self.cfg.patience:
                print(f"Early stopping triggered after {self.cfg.patience} epochs without improvement.")
                break
            
        # Final Test Set Evaluation
        print("\n========== Commencing Final Test Set Evaluation ===========")
        self.model.load_state_dict(torch.load(save_path))
        test_loss, test_auc, test_f1, test_precision, test_recall = self.evaluate(self.test_loader, desc='Testing')
        
        wandb.log({
            "test/loss": test_loss, "test/auc": test_auc, "test/f1": test_f1,
            "test/precision": test_precision, "test/recall": test_recall 
            
        })
        
        print(f"TEST -> Loss: {test_loss:.4f} | AUC: {test_auc:.4f} | F1: {test_f1} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}")

print("Defined the custom CIHMLC Trainer.")

Defined the custom CIHMLC Trainer.


## Section 8: Patient-Aware Data Split & Launch
Executes a double `GroupShuffleSplit` on the patient IDs. First, splitting 70% for Training and 30% for a Temporary set. The Temporary set is split in half to yield the final 15% Validation and 15% Testing sets, ensuring absolute data integrity. <br>
Initializes WandB and executes the trainer.

In [ ]:
import os
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupShuffleSplit
import wandb
from dataclasses import asdict
from kaggle_secrets import UserSecretsClient


def main():
    # Instantiate the global configuration
    CFG = CXRConfig()
    print(f"Executing MediScanX CXR Pipeline on: {CFG.device}")
    
    # Ensure wandb is authenticated in your environment
    from kaggle_secrets import UserSecretsClient
    try:
        user_secrets = UserSecretsClient()
        wandb_api = user_secrets.get_secret("wandb_api_key")
        wandb.login(key=wandb_api)
        print("Successfully authenticated with Weights & Biases!")
    except Exception as e:
        print(f"WandB Auth Error: {e}. Ensure you added WANDB_API_KEY to Kaggle Secrets.")
    
    # Patient-Aware 70/15/15 Split
    print("Executing patient-aware data splitting...")
    df_full = pd.read_csv(CFG.csv_train_path)
    df_full['Patient_ID'] = df_full['Path'].apply(lambda x: x.split('/')[2])
    
    # Split 1: Train and Temp (Val + Test) 
    gss1 = GroupShuffleSplit(n_splits=1, train_size=CFG.train_size, random_state=CFG.random_seed)
    train_idx, temp_idx = next(gss1.split(df_full, groups=df_full['Patient_ID']))
    
    df_train = df_full.iloc[train_idx].copy()
    df_temp = df_full.iloc[temp_idx].copy()
    
    # Split 2: 50% Val, 50% Test from the Temp Set
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=CFG.random_seed)
    val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['Patient_ID']))
    
    df_val = df_temp.iloc[val_idx].copy()
    df_test = df_temp.iloc[test_idx].copy()
    
    # Save splits temporarily for the Dataset class to load
    temp_dir = './temp_splits'
    os.makedirs(temp_dir, exist_ok=True)
    df_train.to_csv(f'{temp_dir}/train_split.csv', index=False)
    df_val.to_csv(f'{temp_dir}/val_split.csv', index=False)
    df_test.to_csv(f'{temp_dir}/test_split.csv', index=False)
    
    # Initialize DataLoaders
    cxr_transforms = RadiographicPipeline.get_training_transforms(resize_dim=CFG.image_size)
    
    train_dataset = CheXpertDataset(f'{temp_dir}/train_split.csv', CFG.kaggle_data_root, transform=cxr_transforms)
    val_dataset = CheXpertDataset(f'{temp_dir}/val_split.csv', CFG.kaggle_data_root, transform=cxr_transforms)
    test_dataset = CheXpertDataset(f'{temp_dir}/test_split.csv', CFG.kaggle_data_root, transform=cxr_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)
    val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
    test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
    
    # Initialize WandB
    wandb.init(
        project=CFG.project_name,
        name=CFG.run_name,
        config=asdict(CFG)
    )
    
    # Architecture, Loss, and Optimizer Setup
    model = DenseNet121_CIHMLC(num_classes=CFG.num_classes, pretrained=CFG.pretrained).to(CFG.device)
    
    pos_weights = ClassWeightCalculator.compute_pos_weights(df_train, num_classes=CFG.num_classes).to(CFG.device)
    hbce_criterion = HBCELoss(pos_weight=pos_weights, hierarchy_pairs=HIERARCHY_PAIRS, penalty_weight=CFG.penalty_weight)
    
    optimizer = optim.Adam(model.parameters(), lr=CFG.learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)
    
    # Execute Training
    trainer = CIHMLCTrainer(
        model=model, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        criterion=hbce_criterion, optimizer=optimizer, scheduler=scheduler, config=CFG
    )
    
    trainer.execute_training(save_path="best_densenet_cihmlc.pth")
    wandb.finish()

if __name__ == "__main__":
    main()

Executing MediScanX CXR Pipeline on: cpu
Executing patient-aware data splitting...



========== Epoch 1/40 ===========


Training:   0%|          | 1/4895 [00:21<28:53:00, 21.25s/it, loss=1.28]